# 9.0 Publication Text Analysis - step 0: lab-data-only benchmark

Before publication text analysis, we want to know whether we can predict a lab's energy use using only administrative data: no_researchers, faculty, and institute. This is the baseline that we will compare the results of our publication text analysis to. 

This notebook runs the following regressions:
1. High/low energy indicator on no_researchers and faculty
2. Continuous energy var on no_researchers and faculty
3. High/low energy indicator on no_researchers and institute
4. Continuous energy var on no_researchers and institute

In [1]:
# Set up
import pandas as pd
import numpy as np
import sys
from pathlib import Path
CODE_ROOT = Path.cwd().parents[1]
sys.path.append(str(CODE_ROOT))
import config
from sklearn.linear_model import LogisticRegressionCV, LassoCV
from sklearn.model_selection import cross_val_score, StratifiedKFold, KFold

In [ ]:
# Load data
labs = pd.read_csv(
    config.CLEAN_DATA / "final_dataset.csv", 
    keep_default_na=False, # Keep "None" as a string, not NaN
    na_values=[""] # Only treat empty strings as NaN
)

publications = pd.read_csv(
    config.PUBLICATON_DATA / 
    "2_Processed" / 
    "publications_matched.csv"
)

## (1) Prepare labs data

In [3]:
# Restrict to BL observations
bl_data = labs[labs["survey"] == "BL"].copy()

# Keep only energy and admin vars cols
bl_data = bl_data[["labgroupid", "annual_electricity_total", "no_researchers", 
                   "faculty", "institute_id"]].copy()

In [ ]:
# Restrict to labs with at least one matched publication (we use this as benchmark)
matched_labgroupids = (
    publications["matched_labgroupids"].astype(str).str.split(";").explode().str.strip().unique()
)

print(f"BL labs before restricting to matched publications: {len(bl_data)}")
bl_data = bl_data[bl_data["labgroupid"].astype(str).isin(matched_labgroupids)].copy()
print(f"BL labs with at least one matched publication:       {len(bl_data)}")

BL labs before restricting to matched publications: 138
BL labs with at least one matched publication:       95


In [5]:
# Prepare variables
median_energy = bl_data["annual_electricity_total"].median()
bl_data["energy_tier"] = np.where(bl_data["annual_electricity_total"] >= median_energy, 1, 0)
bl_data["log_energy"] = np.log(bl_data["annual_electricity_total"])
bl_data["log_no_researchers"] = np.log(bl_data["no_researchers"])

In [6]:
# Check distribution of faculty
print(bl_data["faculty"].value_counts())

# Check distribution of institute_id
print(bl_data["institute_id"].value_counts())

faculty
Faculty of Science (MNF)     71
Faculty of Medicine (MeF)    19
Both MNF and MeF              5
Name: count, dtype: int64
institute_id
1037    13
1025    13
1015    10
1058     9
1064     4
1042     4
1039     4
1005     4
1074     3
1072     3
1054     2
1050     2
1047     2
1080     2
1076     2
1022     2
1067     2
1094     2
1013     2
1056     1
1057     1
1063     1
1077     1
1068     1
1020     1
1038     1
1026     1
1048     1
1021     1
Name: count, dtype: int64


In [7]:
# Create faculty groupings (only science vs. not only science)
bl_data["science_faculty"] = np.where(bl_data["faculty"] == "Faculty of Science (MNF)", 1, 0)

In [8]:
# Create institute groupings (we have many institutes with 1/2 labs each)

# Find small institutes (< 3 labs each)
institute_counts = bl_data["institute_id"].value_counts()
small_institutes = institute_counts[institute_counts < 3].index

# Group small institutes into "Other_science" and "Other_nonscience"

# If institute_id is in small_institutes and science_faculty is 1, then "Other_science", otherwise "Other_nonscience"
bl_data["institute_grouped"] = np.where(
    (bl_data["institute_id"].isin(small_institutes)) & (bl_data["science_faculty"] == 1),
    "Other_science",
    np.where(
        (bl_data["institute_id"].isin(small_institutes)) & (bl_data["science_faculty"] == 0),
        "Other_nonscience",
        bl_data["institute_id"].astype(str),
    ),
)

print(f"Institutes before grouping: {bl_data['institute_id'].nunique()}")
print(f"Categories after grouping:  {bl_data['institute_grouped'].nunique()}")
print(bl_data["institute_grouped"].value_counts())

Institutes before grouping: 29
Categories after grouping:  12
institute_grouped
Other_nonscience    14
Other_science       14
1037                13
1025                13
1015                10
1058                 9
1042                 4
1039                 4
1005                 4
1064                 4
1072                 3
1074                 3
Name: count, dtype: int64


## (2) Faculty-only specification

### (2a) Classification: energy_tier ~ no_researchers + science_faculty

In [9]:
features_fac_clf = pd.get_dummies(bl_data[["no_researchers", "science_faculty"]], columns=["science_faculty"], drop_first=True)
X_fac_clf = features_fac_clf.values.astype(float)
y_clf = bl_data["energy_tier"].values

outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
clf = LogisticRegressionCV(Cs=10, cv=5, penalty="l1", solver="liblinear", max_iter=1000, random_state=0)

cv_accuracy = cross_val_score(clf, X_fac_clf, y_clf, cv=outer_cv, scoring="accuracy")
cv_auc = cross_val_score(clf, X_fac_clf, y_clf, cv=outer_cv, scoring="roc_auc")

print(f"Nested CV accuracy: {cv_accuracy.mean():.1%} (+/- {cv_accuracy.std():.1%})")
print(f"Nested CV ROC-AUC:  {cv_auc.mean():.3f} (+/- {cv_auc.std():.3f})")

Nested CV accuracy: 69.5% (+/- 6.1%)
Nested CV ROC-AUC:  0.784 (+/- 0.071)


### (2b) Regression: log(energy) ~ log(no_researchers) + faculty

In [10]:
features_fac_reg = pd.get_dummies(bl_data[["science_faculty"]], columns=["science_faculty"], drop_first=True)
features_fac_reg.insert(0, "log_no_researchers", bl_data["log_no_researchers"].values)
X_fac_reg = features_fac_reg.values.astype(float)
y_reg = bl_data["log_energy"].values

outer_cv_reg = KFold(n_splits=5, shuffle=True, random_state=0)
reg = LassoCV(cv=5, max_iter=5000, random_state=0)

cv_r2 = cross_val_score(reg, X_fac_reg, y_reg, cv=outer_cv_reg, scoring="r2")
cv_rmse = -cross_val_score(reg, X_fac_reg, y_reg, cv=outer_cv_reg, scoring="neg_root_mean_squared_error")

print(f"Nested CV R²:   {cv_r2.mean():.3f} (+/- {cv_r2.std():.3f})")
print(f"Nested CV RMSE: {cv_rmse.mean():.3f} (+/- {cv_rmse.std():.3f})")

Nested CV R²:   0.310 (+/- 0.046)
Nested CV RMSE: 1.355 (+/- 0.180)


In [ ]:
# Fit on the full 95 labs for interpretation of coefficients
reg_fac_full = LassoCV(cv=5, max_iter=5000, random_state=0)
reg_fac_full.fit(X_fac_reg, y_reg)

coef_df = pd.DataFrame({"feature": features_fac_reg.columns, "coefficient": reg_fac_full.coef_})
print(f"Selected alpha: {reg_fac_full.alpha_:.4f}")
print(coef_df.to_string(index=False))

Selected alpha: 0.0061
           feature  coefficient
log_no_researchers     1.476016
 science_faculty_1    -0.466295


## (3) Institute specification

### (3a) Classification: energy_tier ~ no_researchers + institute_grouped

In [12]:
features_inst_clf = pd.get_dummies(
    bl_data[["no_researchers", "institute_grouped"]], columns=["institute_grouped"], drop_first=True
)
X_inst_clf = features_inst_clf.values.astype(float)

cv_accuracy_inst = cross_val_score(clf, X_inst_clf, y_clf, cv=outer_cv, scoring="accuracy")
cv_auc_inst = cross_val_score(clf, X_inst_clf, y_clf, cv=outer_cv, scoring="roc_auc")

print(f"Nested CV accuracy: {cv_accuracy_inst.mean():.1%} (+/- {cv_accuracy_inst.std():.1%})")
print(f"Nested CV ROC-AUC:  {cv_auc_inst.mean():.3f} (+/- {cv_auc_inst.std():.3f})")

Nested CV accuracy: 73.7% (+/- 13.3%)
Nested CV ROC-AUC:  0.827 (+/- 0.078)


### (3b) Regression: log(energy) ~ log(no_researchers) + institute_grouped

In [13]:
features_inst_reg = pd.get_dummies(
    bl_data[["institute_grouped"]], columns=["institute_grouped"], drop_first=True
)
features_inst_reg.insert(0, "log_no_researchers", bl_data["log_no_researchers"].values)
X_inst_reg = features_inst_reg.values.astype(float)

cv_r2_inst = cross_val_score(reg, X_inst_reg, y_reg, cv=outer_cv_reg, scoring="r2")
cv_rmse_inst = -cross_val_score(reg, X_inst_reg, y_reg, cv=outer_cv_reg, scoring="neg_root_mean_squared_error")

print(f"Nested CV R²:   {cv_r2_inst.mean():.3f} (+/- {cv_r2_inst.std():.3f})")
print(f"Nested CV RMSE: {cv_rmse_inst.mean():.3f} (+/- {cv_rmse_inst.std():.3f})")

Nested CV R²:   0.489 (+/- 0.059)
Nested CV RMSE: 1.166 (+/- 0.167)


In [ ]:
# Fit on the full 95 labs for interpretation of coefficients
reg_inst_full = LassoCV(cv=5, max_iter=5000, random_state=0)
reg_inst_full.fit(X_inst_reg, y_reg)

coef_df_inst = pd.DataFrame({"feature": features_inst_reg.columns, "coefficient": reg_inst_full.coef_})
coef_df_inst["abs_coef"] = coef_df_inst["coefficient"].abs()
n_nonzero = (coef_df_inst["coefficient"] != 0).sum()

print(f"Selected alpha: {reg_inst_full.alpha_:.4f}")
print(f"Features with non-zero coefficient: {n_nonzero} / {len(features_inst_reg.columns)}")
print(coef_df_inst.sort_values("abs_coef", ascending=False).drop(columns="abs_coef").to_string(index=False))

Selected alpha: 0.0265
Features with non-zero coefficient: 6 / 12
                           feature  coefficient
                log_no_researchers     1.395607
            institute_grouped_1058    -1.170071
            institute_grouped_1025     0.978917
            institute_grouped_1037    -0.702811
   institute_grouped_Other_science    -0.495359
            institute_grouped_1064     0.385687
            institute_grouped_1015    -0.000000
            institute_grouped_1039    -0.000000
            institute_grouped_1042     0.000000
            institute_grouped_1072     0.000000
            institute_grouped_1074     0.000000
institute_grouped_Other_nonscience    -0.000000


We see that institute has more predictive power than faculty alone.